In [1]:
# Constantes de ronda oficiales para Keccak-f (24 rondas)
RC = [
    0x0000000000000001, 0x0000000000008082, 0x800000000000808A, 0x8000000080008000,
    0x000000000000808B, 0x0000000080000001, 0x8000000080008081, 0x8000000000008009,
    0x000000000000008A, 0x0000000000000088, 0x0000000080008009, 0x000000008000000A,
    0x000000008000808B, 0x800000000000008B, 0x8000000000008089, 0x8000000000008003,
    0x8000000000008002, 0x8000000000000080, 0x000000000000800A, 0x800000008000000A,
    0x8000000080008081, 0x8000000000008080, 0x0000000080000001, 0x8000000080008008
]

# Matriz de rotaciones fijas para el paso Rho
ROTATION_OFFSETS = [
    [0,  36,  3,  41, 18],
    [1,  44, 10,  45,  2],
    [62,  6, 43,  15, 61],
    [28, 55, 25,  21, 56],
    [27, 20, 39,   8, 14]
]

def ROTL64(x, n):
    """Rotación de bits circular para un entero de 64 bits."""
    return ((x << (n % 64)) & 0xFFFFFFFFFFFFFFFF) | (x >> (64 - (n % 64)))

def keccak_f1600(state):
    """Permutación nativa Keccak-f sobre una matriz de 5x5 enteros de 64 bits."""
    for round_idx in range(24):
        # 1. Paso Theta (θ)
        C = [0] * 5
        for x in range(5):
            C[x] = state[x][0] ^ state[x][1] ^ state[x][2] ^ state[x][3] ^ state[x][4]

        D = [0] * 5
        for x in range(5):
            D[x] = C[(x - 1) % 5] ^ ROTL64(C[(x + 1) % 5], 1)

        for x in range(5):
            for y in range(5):
                state[x][y] ^= D[x]

        # 2. Pasos Rho (ρ) y Pi (π) combinados
        next_state = [[0] * 5 for _ in range(5)]
        for x in range(5):
            for y in range(5):
                next_state[y][(2 * x + 3 * y) % 5] = ROTL64(state[x][y], ROTATION_OFFSETS[x][y])
        state = next_state

        # 3. Paso Chi (χ)
        next_state = [[0] * 5 for _ in range(5)]
        for x in range(5):
            for y in range(5):
                next_state[x][y] = state[x][y] ^ ((~state[(x + 1) % 5][y]) & state[(x + 2) % 5][y])
        state = next_state

        # 4. Paso Iota (ι) - Afecta únicamente al carril de origen (0,0)
        state[0][0] ^= RC[round_idx]

    return state

def sha3_256_fips(message: bytes) -> bytes:
    """Implementación exacta de SHA3-256 según la especificación estándar FIPS 202."""
    rate_bytes = 136  # r = 1088 bits para SHA3-256
    state = [[0] * 5 for _ in range(5)]

    # Construcción matemática del relleno (Padding pad10*1)
    # Se concatena el sufijo del dominio SHA-3 (b'01') junto al bit de inicio b'1' -> 0x06
    padded = bytearray(message)
    padded.append(0x06)

    # Rellenar con ceros intermedios hasta completar el bloque
    while len(padded) % rate_bytes != 0:
        padded.append(0x00)

    # El bit final del padding pad10*1 se activa en el bit más significativo
    padded[-1] |= 0x80

    # Fase de Absorción
    for block_idx in range(0, len(padded), rate_bytes):
        block = padded[block_idx:block_idx + rate_bytes]
        for i in range(rate_bytes // 8):
            x = i % 5
            y = i // 5
            word = int.from_bytes(block[i*8:(i+1)*8], 'little')
            state[x][y] ^= word
        state = keccak_f1600(state)

    # Fase de Exprimido (Squeezing) - Extraemos 32 bytes (256 bits)
    output = bytearray()
    for i in range(4):  # 4 carriles de 8 bytes = 32 bytes
        x = i % 5
        y = i // 5
        output.extend(state[x][y].to_bytes(8, 'little'))

    return bytes(output)


if __name__ == "__main__":
    import hashlib

    casos_prueba = [
        b"",
        b"abc",
        b"El cifrado Keccak cumple con el estandar FIPS 202 sin problemas de compatibilidad.",
        b"a" * 135,  # Frontera de bloque - 1 byte
        b"a" * 136,  # Frontera exacta de un bloque
        b"a" * 137   # Frontera de bloque + 1 byte
    ]

    print("--- VERIFICACIÓN FIPS 202 ---")
    for msg in casos_prueba:
        hash_propio = sha3_256_fips(msg).hex()
        hash_oficial = hashlib.sha3_256(msg).hexdigest()

        print(f"Mensaje len({len(msg)}): {msg[:20]}...")
        print(f"  > Propio:  {hash_propio}")
        print(f"  > Oficial: {hash_oficial}")

        assert hash_propio == hash_oficial, f"¡Error : {msg}!"

    print("\n¡hash  coinciden  con el estándar FIPS.")

--- VERIFICACIÓN FIPS 202 ---
Mensaje len(0): b''...
  > Propio:  a7ffc6f8bf1ed76651c14756a061d662f580ff4de43b49fa82d80a4b80f8434a
  > Oficial: a7ffc6f8bf1ed76651c14756a061d662f580ff4de43b49fa82d80a4b80f8434a
Mensaje len(3): b'abc'...
  > Propio:  3a985da74fe225b2045c172d6bd390bd855f086e3e9d525b46bfe24511431532
  > Oficial: 3a985da74fe225b2045c172d6bd390bd855f086e3e9d525b46bfe24511431532
Mensaje len(82): b'El cifrado Keccak cu'...
  > Propio:  75b132c9693574d34dc6c0f87c9b5a5ed381f43d8e80d656698cb33daeeb99a6
  > Oficial: 75b132c9693574d34dc6c0f87c9b5a5ed381f43d8e80d656698cb33daeeb99a6
Mensaje len(135): b'aaaaaaaaaaaaaaaaaaaa'...
  > Propio:  8094bb53c44cfb1e67b7c30447f9a1c33696d2463ecc1d9c92538913392843c9
  > Oficial: 8094bb53c44cfb1e67b7c30447f9a1c33696d2463ecc1d9c92538913392843c9
Mensaje len(136): b'aaaaaaaaaaaaaaaaaaaa'...
  > Propio:  3fc5559f14db8e453a0a3091edbd2bc25e11528d81c66fa570a4efdcc2695ee1
  > Oficial: 3fc5559f14db8e453a0a3091edbd2bc25e11528d81c66fa570a4efdcc2695ee1
Mensaje

In [2]:
# Constantes de ronda para Keccak-f (24 rondas)
RC = [
    0x0000000000000001, 0x0000000000008082, 0x800000000000808A, 0x8000000080008000,
    0x000000000000808B, 0x0000000080000001, 0x8000000080008081, 0x8000000000008009,
    0x000000000000008A, 0x0000000000000088, 0x0000000080008009, 0x000000008000000A,
    0x000000008000808B, 0x800000000000008B, 0x8000000000008089, 0x8000000000008003,
    0x8000000000008002, 0x8000000000000080, 0x000000000000800A, 0x800000008000000A,
    0x8000000080008081, 0x8000000000008080, 0x0000000080000001, 0x8000000080008008
]

# Desplazamientos de rotación específicos para cada celda de la matriz 5x5
ROTATION_OFFSETS = [
    [0, 36, 3, 41, 18],
    [1, 44, 10, 45, 2],
    [62, 6, 43, 15, 61],
    [28, 55, 25, 21, 56],
    [27, 20, 39, 8, 14]
]

def ROTL64(x, n):
    """Rotación circular a la izquierda de 64 bits."""
    return ((x << (n % 64)) & 0xFFFFFFFFFFFFFFFF) | (x >> (64 - (n % 64)))

def keccak_f1600(state):
    """Ejecuta las 24 rondas sobre la matriz de estado."""
    for round_idx in range(24):
        # 1. Paso Theta (θ)
        C = [0] * 5
        for x in range(5):
            C[x] = state[x][0] ^ state[x][1] ^ state[x][2] ^ state[x][3] ^ state[x][4]

        D = [0] * 5
        for x in range(5):
            D[x] = C[(x - 1) % 5] ^ ROTL64(C[(x + 1) % 5], 1)

        for x in range(5):
            for y in range(5):
                state[x][y] ^= D[x]

        # 2. Pasos Rho (ρ) y Pi (π) combinados
        next_state = [[0] * 5 for _ in range(5)]
        for x in range(5):
            for y in range(5):
                next_state[y][(2 * x + 3 * y) % 5] = ROTL64(state[x][y], ROTATION_OFFSETS[x][y])
        state = next_state

        # 3. Paso Chi (χ)
        next_state = [[0] * 5 for _ in range(5)]
        for x in range(5):
            for y in range(5):
                next_state[x][y] = state[x][y] ^ ((~state[(x + 1) % 5][y]) & state[(x + 2) % 5][y])
        state = next_state

        # 4. Paso Iota (ι) afecta sólo a la celda de origen [0][0]
        state[0][0] ^= RC[round_idx]

    return state

def sha3_256(message: bytes) -> bytes:
    """Función de Esponja SHA3-256"""
    rate_bytes = 136  # r = 1088 bits para SHA3-256
    state = [[0] * 5 for _ in range(5)]

    # Relleno oficial pad10*1 junto al dominio b'01' de SHA-3
    padded = bytearray(message)
    padded.append(0x06)
    while (len(padded) % rate_bytes) != (rate_bytes - 1):
        padded.append(0x00)
    padded.append(0x80)

    # Fase de Absorción
    for block_idx in range(0, len(padded), rate_bytes):
        block = padded[block_idx:block_idx + rate_bytes]
        for i in range(rate_bytes // 8):
            x = i % 5
            y = i // 5
            word = int.from_bytes(block[i*8:(i+1)*8], 'little')
            state[x][y] ^= word
        state = keccak_f1600(state)

    # Fase de Exprimido (Squeezing) - Extraer los 32 bytes del Hash
    output = bytearray()
    for i in range(4):  # 4 palabras * 8 bytes = 32 bytes (256 bits)
        x = i % 5
        y = i // 5
        output.extend(state[x][y].to_bytes(8, 'little'))

    return bytes(output)


intentos_fallidos = 0

def registrar_intento_fallido():
    """Incrementa el contador global de intentos fallidos."""
    global intentos_fallidos
    intentos_fallidos += 1


# FR-008: segunda variable dinamica (dominio de la aplicacion, objetivo 1),
# ademas de intentos_fallidos (nivel de seguridad). Se representa como
# un entero de dominio cerrado 0-3: 0 = IoT/sensor, 1 = tarjeta inteligente, 2 = firmware embebido,
# 3 = general. Se combina SOLO por XOR en iota, igual que intentos_fallidos, para
# no reabrir una superficie de ataque ya cerrada por el analisis MILP existente.
DOMINIOS_APLICACION = {
    0: "IoT/sensor",
    1: "tarjeta inteligente",
    2: "firmware embebido",
    3: "general",
}
dominio_aplicacion = 0

def establecer_dominio_aplicacion(valor):
    """Fija la variable dinamica de dominio de la aplicacion (0-3)."""
    global dominio_aplicacion
    if valor not in DOMINIOS_APLICACION:
        raise ValueError(f"dominio_aplicacion debe estar en {sorted(DOMINIOS_APLICACION)}")
    dominio_aplicacion = valor


# Pasos individuales de Keccak SEPARADOS para poder reordenarlos dinámicamente.
def theta(state):
    """Paso Theta (θ): difusión de columnas mediante XOR."""
    C = [0] * 5
    for x in range(5):
        C[x] = state[x][0] ^ state[x][1] ^ state[x][2] ^ state[x][3] ^ state[x][4]
    D = [0] * 5
    for x in range(5):
        D[x] = C[(x - 1) % 5] ^ ROTL64(C[(x + 1) % 5], 1)
    for x in range(5):
        for y in range(5):
            state[x][y] ^= D[x]
    return state

def rho(state):
    """Paso Rho (ρ): solo rotaciones de cada carril, sin reubicar (separado de Pi)."""
    for x in range(5):
        for y in range(5):
            state[x][y] = ROTL64(state[x][y], ROTATION_OFFSETS[x][y])
    return state

def pi(state):
    """Paso Pi (π): solo reubicación/permutación de las celdas (separado de Rho)."""
    next_state = [[0] * 5 for _ in range(5)]
    for x in range(5):
        for y in range(5):
            next_state[y][(2 * x + 3 * y) % 5] = state[x][y]
    return next_state

def chi(state):
    """Paso Chi (χ): no linealidad mediante AND/NOT por filas."""
    next_state = [[0] * 5 for _ in range(5)]
    for x in range(5):
        for y in range(5):
            next_state[x][y] = state[x][y] ^ ((~state[(x + 1) % 5][y]) & state[(x + 2) % 5][y])
    return next_state

def sigma(state, intentos_fallidos):
    """Transformación experimental dependiente de intentos_fallidos.
    Aplica un XOR y una pequeña rotación de bits para introducir variabilidad
    dinámica adicional. """
    for x in range(5):
        for y in range(5):
            # XOR dependiente del contexto (intentos_fallidos) y de la posición
            state[x][y] ^= ((intentos_fallidos << (x + y)) & 0xFFFFFFFFFFFFFFFF)
            # Pequeña rotación adicional dependiente de intentos_fallidos
            state[x][y] = ROTL64(state[x][y], (intentos_fallidos + x + y) % 64)
    return state

def iota(state, round_idx, intentos_fallidos, dominio_aplicacion=0):
    """Paso Iota (ι) con constante de ronda dependiente del contexto.
    La constante RC depende de intentos_fallidos (nivel de seguridad) y, desde
    FR-008, tambien de dominio_aplicacion (dominio de la aplicacion). Ambas
    variables entran solo como XOR de una constante conocida (no dependen del
    mensaje), por lo que no alteran la propagacion de una diferencial, ver
    README.md, iota dinamica, y el analisis MILP correspondiente."""
    RC_modificada = RC[round_idx] ^ intentos_fallidos ^ dominio_aplicacion
    state[0][0] ^= RC_modificada
    return state


def keccak_f1600_variante(state):
    """Permutación Keccak-f adaptativa controlada por intentos_fallidos (rondas y
    orden de pasos) y, desde FR-008, por dominio_aplicacion (solo dentro de iota;
    NO afecta rondas ni orden -- dominio_aplicacion no aparece en la formula de
    `rondas` ni en la condicion `if intentos_fallidos < 3`, por diseno)."""
    # A mayor número de intentos, el número de rondas se acerca progresivamente a 24
    rondas = min(24, 8 + intentos_fallidos)
    for round_idx in range(rondas):
        # Se altera SOLO el pipeline de ejecución para demostrar experimentalmente
        # la NO CONMUTATIVIDAD de las transformaciones de Keccak.
        if intentos_fallidos < 3:
            # Orden original: theta -> rho -> pi -> chi -> iota
            state = theta(state)
            state = rho(state)
            state = pi(state)
            state = chi(state)
            state = iota(state, round_idx, intentos_fallidos, dominio_aplicacion)
        else:
            # Orden experimental: theta -> rho -> chi -> pi -> sigma -> iota
            state = theta(state)
            state = rho(state)
            state = chi(state)
            state = pi(state)
            state = sigma(state, intentos_fallidos)
            state = iota(state, round_idx, intentos_fallidos, dominio_aplicacion)
    return state


# Objetivo 1: dominio_aplicacion tambien selecciona el par
# (ratio, capacidad) de la construccion esponja, ademas de su rol ya validado
# dentro de iota. Los 4 perfiles son pares (rate, capacity) OFICIALES de la
# familia SHA3 (no valores inventados): equivalen a SHA3-256/224/384/512. El
# dominio 0 mantiene el par estandar de SHA3-256 (136 bytes / 512 bits) para no
# alterar el comportamiento por defecto ya validado en la celda anterior.
PERFILES_DOMINIO = {
    0: {"nombre": "general (por defecto)", "rate_bytes": 136, "capacidad_bits": 512},   # SHA3-256
    1: {"nombre": "IoT / sensor (recursos extremos)", "rate_bytes": 144, "capacidad_bits": 448},  # SHA3-224
    2: {"nombre": "tarjeta inteligente", "rate_bytes": 104, "capacidad_bits": 768},      # SHA3-384
    3: {"nombre": "firmware embebido critico", "rate_bytes": 72, "capacidad_bits": 1024}, # SHA3-512
}

def cota_seguridad_generica_bits(capacidad_bits, salida_bits=256):
    """Cota generica de seguridad de una esponja: min(capacidad/2, longitud de
    salida). Formula estandar de la construccion esponja (Bertoni et al.)."""
    return min(capacidad_bits // 2, salida_bits)

def sha3_256_variante(message: bytes) -> bytes:
    """Variante experimental de SHA3-256 que usa keccak_f1600_variante. El ratio
    (y por tanto la capacidad, ratio+capacidad=1600 bits) ahora depende de
    dominio_aplicacion (ver PERFILES_DOMINIO); el resto de la sponge construction
    se mantiene identico a la original."""
    rate_bytes = PERFILES_DOMINIO[dominio_aplicacion]["rate_bytes"]
    state = [[0] * 5 for _ in range(5)]

    padded = bytearray(message)
    padded.append(0x06)
    while len(padded) % rate_bytes != 0:
        padded.append(0x00)
    padded[-1] |= 0x80

    for block_idx in range(0, len(padded), rate_bytes):
        block = padded[block_idx:block_idx + rate_bytes]
        for i in range(rate_bytes // 8):
            x = i % 5
            y = i // 5
            word = int.from_bytes(block[i*8:(i+1)*8], 'little')
            state[x][y] ^= word
        state = keccak_f1600_variante(state)

    output = bytearray()
    for i in range(4):
        x = i % 5
        y = i // 5
        output.extend(state[x][y].to_bytes(8, 'little'))

    return bytes(output)


if __name__ == "__main__":
    # Misma entrada de texto para ambos cálculos
    mensaje = b"password123"

    # 1) Hash original Keccak/SHA3-256 estándar
    hash_original = sha3_256_fips(mensaje).hex()

    # Simulamos varios intentos fallidos para activar el orden experimental
    # (intentos_fallidos >= 3) y las rondas/constantes dinámicas.
    for _ in range(5):
        registrar_intento_fallido()

    # 2) Hash de la variante adaptativa
    hash_variante = sha3_256_variante(mensaje).hex()

    print("\n--- COMPARACIÓN EXPERIMENTAL (NO CONMUTATIVIDAD) ---")
    print(f"intentos_fallidos = {intentos_fallidos}")
    print(f"Hash original:  {hash_original}")
    print(f"Hash variante:  {hash_variante}")
    # Los hashes son distintos: cambiar el orden de composición de los pasos
    # altera el resultado. Esto es un contraejemplo práctico de NO CONMUTATIVIDAD.
    print("Los hashes son DISTINTOS -> cambiar el orden de composición altera el resultado.")


--- COMPARACIÓN EXPERIMENTAL (NO CONMUTATIVIDAD) ---
intentos_fallidos = 5
Hash original:  ab3fe4003f14e3ef573417f95e47d4985c482eadd139c08b3758eeae7cc60b9d
Hash variante:  a5f8cd24080e6d56b242b7b9340d9c1af7274d41820e0a34b7cf906d9746cb33
Los hashes son DISTINTOS -> cambiar el orden de composición altera el resultado.


In [3]:
# --- FR-008: verificacion de la segunda variable dinamica (dominio_aplicacion) ---
# Requiere haber ejecutado la celda anterior (define keccak_f1600_variante,
# sha3_256_variante, mensaje, intentos_fallidos=5, DOMINIOS_APLICACION, etc.)

# T006 (no-regresion): con dominio_aplicacion=0 (valor por defecto) el resultado
# debe ser IDENTICO al capturado antes de FR-008.
establecer_dominio_aplicacion(0)
hash_variante_regresion = sha3_256_variante(mensaje).hex()
HASH_BASELINE_SIN_DOMINIO = "a5f8cd24080e6d56b242b7b9340d9c1af7274d41820e0a34b7cf906d9746cb33"
assert hash_variante_regresion == HASH_BASELINE_SIN_DOMINIO, (
    f"Regresion FR-008: con dominio_aplicacion=0 el hash cambio de "
    f"{HASH_BASELINE_SIN_DOMINIO} a {hash_variante_regresion}"
)
print(f"[T006] Sin regresion: dominio_aplicacion=0 reproduce el hash baseline "
      f"({hash_variante_regresion}).")

# T011 (rondas y orden invariantes): por construccion, dominio_aplicacion no
# aparece en la formula de `rondas` ni en la condicion de orden de
# keccak_f1600_variante (ver su codigo fuente arriba) -- solo se lee dentro de
# iota. Con intentos_fallidos=5 fijo, las rondas (13) y el orden (experimental,
# >=3) son los mismos para cualquier dominio_aplicacion.
print(f"[T011] intentos_fallidos={intentos_fallidos} -> rondas=min(24,8+{intentos_fallidos})="
      f"{min(24, 8 + intentos_fallidos)} y orden=experimental, iguales para los 4 dominios "
      f"(dominio_aplicacion no participa en su calculo).")

# T012 (efecto observable, distinto por dominio): variar SOLO dominio_aplicacion,
# con el mismo mensaje e intentos_fallidos, debe producir hashes distintos entre si
# (evidencia de que el termino XOR adicional en iota si tiene efecto).
hashes_por_dominio = {}
for dominio, nombre in DOMINIOS_APLICACION.items():
    establecer_dominio_aplicacion(dominio)
    h = sha3_256_variante(mensaje).hex()
    hashes_por_dominio[dominio] = h
    print(f"[T012] dominio_aplicacion={dominio} ({nombre}): hash={h}")

assert len(set(hashes_por_dominio.values())) == len(hashes_por_dominio), (
    "T012: se esperaban 4 hashes distintos (uno por dominio_aplicacion), "
    "se encontraron colisiones"
)
print("[T012] Los 4 valores de dominio_aplicacion producen hashes distintos entre si.")

establecer_dominio_aplicacion(0)  # deja el estado global en el valor por defecto
print("\nFR-008 verificado: sin regresion, rondas/orden invariantes, efecto "
      "observable y distinto por dominio.")


[T006] Sin regresion: dominio_aplicacion=0 reproduce el hash baseline (a5f8cd24080e6d56b242b7b9340d9c1af7274d41820e0a34b7cf906d9746cb33).
[T011] intentos_fallidos=5 -> rondas=min(24,8+5)=13 y orden=experimental, iguales para los 4 dominios (dominio_aplicacion no participa en su calculo).
[T012] dominio_aplicacion=0 (IoT/sensor): hash=a5f8cd24080e6d56b242b7b9340d9c1af7274d41820e0a34b7cf906d9746cb33
[T012] dominio_aplicacion=1 (tarjeta inteligente): hash=64a86d5cc3f491ac00b6a0084aee9ddcb7b75543b21d3811a217ff818c137828
[T012] dominio_aplicacion=2 (firmware embebido): hash=bd640487332adcd14970f482d560f5b5c8ae173736b3b575dad9c1ae06353667
[T012] dominio_aplicacion=3 (general): hash=66ce01de6db1c0fb24525ecf63f3adeaf581c0cd820a78404f5ecb9fa4478781
[T012] Los 4 valores de dominio_aplicacion producen hashes distintos entre si.

FR-008 verificado: sin regresion, rondas/orden invariantes, efecto observable y distinto por dominio.


In [4]:
# --- Objetivos 2 y 3: rediseno de la capa lineal y del paso no lineal ---
# Requiere haber ejecutado la celda anterior (RC, ROTATION_OFFSETS, ROTL64, theta,
# rho, pi, chi, iota, mensaje, hash_original, hash_variante, etc.)

# ============================================================================
# PUNTO 2: Rediseno de la capa lineal (reemplaza theta+rho+pi)
# ============================================================================
# Nueva funcion de difusion, construida ESTRICTAMENTE con XOR y rotaciones (sin
# AND/OR). Idea: hacer lo mismo que theta (reducir una columna a un valor y
# difundirlo) pero TAMBIEN para filas (dimension que theta no mezcla en un solo
# paso), y -- a diferencia de un primer intento ingenuo que probamos y descartamos
# por diluirse a una fraccion fija de bits sin importar el ancho de palabra --
# reutilizar ROTATION_OFFSETS para que el valor reducido de cada columna/fila se
# difunda con una rotacion DISTINTA en cada celda (x,y), igual que hace rho con
# el estado completo. Esto asegura que un solo bit alcance TODAS las lanas con
# una rotacion distinta cada vez, en vez de repetir la misma posicion de bit.

def theta_columnas(state):
    """Como theta: reduce cada columna (eje X) a un valor C[x] y lo difunde,
    pero con una rotacion DISTINTA por cada celda (x,y) via ROTATION_OFFSETS,
    en vez de la misma rotacion para las 5 lanas de la columna destino."""
    state = [fila[:] for fila in state]
    C = [0] * 5
    for x in range(5):
        v = 0
        for y in range(5):
            v ^= state[x][y]
        C[x] = v
    D_base = [C[(x - 1) % 5] ^ ROTL64(C[(x + 1) % 5], 1) for x in range(5)]
    for x in range(5):
        for y in range(5):
            state[x][y] ^= ROTL64(D_base[x], ROTATION_OFFSETS[x][y])
    return state

def theta_filas(state):
    """Analogo a theta_columnas pero mezclando FILAS (eje Y): la dimension que
    theta original no difunde en un solo paso."""
    state = [fila[:] for fila in state]
    E = [0] * 5
    for y in range(5):
        v = 0
        for x in range(5):
            v ^= state[x][y]
        E[y] = v
    F_base = [E[(y - 1) % 5] ^ ROTL64(E[(y + 1) % 5], 2) for y in range(5)]
    for x in range(5):
        for y in range(5):
            state[x][y] ^= ROTL64(F_base[y], ROTATION_OFFSETS[x][y])
    return state

def capa_lineal_rediseno(state):
    """Reemplazo propuesto de theta+rho+pi: difusion de columna seguida de
    difusion de fila, aplicadas en secuencia sobre el estado ya actualizado."""
    return theta_filas(theta_columnas(state))


# --- Verificacion de invertibilidad (requisito necesario: Keccak-f debe ser una
# permutacion). Se construye la matriz GF(2) de 1600x1600 aplicando la funcion a
# cada vector base y se verifica que tenga rango completo (Gauss-Jordan con filas
# empaquetadas como enteros grandes, misma tecnica que
# MILP_AES_Keccak_modificado.ipynb pero al ancho de palabra real de 64 bits). ---

def _state_to_flat(state, z=64):
    return [(state[x][y] >> b) & 1 for x in range(5) for y in range(5) for b in range(z)]

def _flat_to_state(bits, z=64):
    state = [[0] * 5 for _ in range(5)]
    idx = 0
    for x in range(5):
        for y in range(5):
            v = 0
            for b in range(z):
                v |= bits[idx] << b
                idx += 1
            state[x][y] = v
    return state

def _rango_gf2(fn, z=64):
    n = 25 * z
    columnas = []
    for i in range(n):
        bits = [0] * n
        bits[i] = 1
        columnas.append(_state_to_flat(fn(_flat_to_state(bits, z)), z))
    filas = []
    for r in range(n):
        m = 0
        for c in range(n):
            if columnas[c][r]:
                m |= 1 << c
        filas.append(m)
    rango = 0
    for col in range(n):
        piv = None
        for r in range(rango, n):
            if (filas[r] >> col) & 1:
                piv = r
                break
        if piv is None:
            continue
        filas[rango], filas[piv] = filas[piv], filas[rango]
        for r in range(n):
            if r != rango and (filas[r] >> col) & 1:
                filas[r] ^= filas[rango]
        rango += 1
    return rango, n

_rango, _n = _rango_gf2(capa_lineal_rediseno)
print(f"[Punto 2] capa_lineal_rediseno: rango GF(2) = {_rango}/{_n} "
      f"({'INVERTIBLE, es una permutacion valida' if _rango == _n else 'NO INVERTIBLE -- invalido para Keccak-f'})")
assert _rango == _n, "capa_lineal_rediseno debe ser invertible para servir de permutacion"


# --- Medicion de difusion a nivel de LANA: para cada uno de los 1600 bits de
# estado, se activa solo ese bit y se mide que fraccion de las 25 lanas del
# estado quedan con AL MENOS un bit activo tras 1-3 aplicaciones (esta es la
# nocion de "difusion" que usa la documentacion original de Keccak: cuantas
# rondas hacen falta para que un bit alcance TODAS las lanas -- no confundir
# con la fraccion de BITS individuales que cambian, que para una funcion lineal
# bien mezclada converge estructuralmente a ~50%, no a 100%, igual que el
# criterio de avalancha de una funcion no lineal ideal). Se compara contra
# theta+rho+pi estandar. ---

def _theta_rho_pi_estandar(state):
    s = theta(state)
    s = rho(s)
    s = pi(s)
    return s

def _difusion_lanas_pct(fn, rondas, z=64):
    n = 25 * z
    total = 0
    minimo = 25
    for i in range(n):
        bits = [0] * n
        bits[i] = 1
        s = _flat_to_state(bits, z)
        for _ in range(rondas):
            s = fn(s)
        lanas_activas = sum(1 for x in range(5) for y in range(5) if s[x][y] != 0)
        total += lanas_activas
        minimo = min(minimo, lanas_activas)
    return total / (n * 25), minimo / 25

print("\n[Punto 2] Difusion a nivel de lana (fraccion de las 25 lanas tocadas), a 64 bits reales:")
for rondas in (1, 2, 3):
    avg_r, min_r = _difusion_lanas_pct(capa_lineal_rediseno, rondas)
    avg_e, min_e = _difusion_lanas_pct(_theta_rho_pi_estandar, rondas)
    print(f"  rondas={rondas}: REDISENO avg={avg_r:.1%} min={min_r:.1%}  |  ESTANDAR (theta+rho+pi) avg={avg_e:.1%} min={min_e:.1%}")

print(
    "\n[Punto 2] Resultado: capa_lineal_rediseno alcanza >=80% de difusion de lanas en "
    "UNA sola ronda (ver tabla), mientras que theta+rho+pi estandar necesita 2-3 rondas "
    "para llegar al mismo nivel, ver README.md para el analisis completo, "
    "incluyendo por que se descarto un primer intento mas simple (misma rotacion para "
    "toda la columna) que no escalaba con el ancho de palabra, y la comparacion de peso "
    "de trail diferencial en MILP_AES_Keccak_modificado.ipynb, que muestra una cota de "
    "resistencia igual en 1 ronda y notablemente MAYOR en 2-3 rondas frente al estandar."
)


# ============================================================================
# PUNTO 3: Rediseno del paso no lineal -- dos intentos de simplificacion,
# AMBOS invalidados con demostracion matematica, ver README.md
# ============================================================================

def _chi_row_biyectiva(chi_row_fn):
    salidas = set()
    for v in range(32):
        bits = tuple((v >> i) & 1 for i in range(5))
        out = chi_row_fn(bits)
        salidas.add(sum(b << i for i, b in enumerate(out)))
    return len(salidas) == 32

def _chi_row_estandar(a):
    return tuple(a[x] ^ ((1 - a[(x + 1) % 5]) & a[(x + 2) % 5]) for x in range(5))

def _chi_row_sin_not(a):
    return tuple(a[x] ^ (a[(x + 1) % 5] & a[(x + 2) % 5]) for x in range(5))

print("\n[Punto 3] Intento 1 -- chi_sin_not (elimina el NOT, 1 compuerta menos por bit):")
print(f"  chi estandar es biyectiva (fuerza bruta 32 casos): {_chi_row_biyectiva(_chi_row_estandar)}")
print(f"  chi_sin_not es biyectiva (fuerza bruta 32 casos):  {_chi_row_biyectiva(_chi_row_sin_not)}")
print("  -> INVALIDO: al no ser biyectiva, keccak_f dejaria de ser una permutacion "
      "(colisiones internas de estado). Se descarta.")


def chi_parcial(state, bits_activos):
    """Intento 2: aplica chi estandar solo a un subconjunto de posiciones de bit
    del carril (bits_activos), dejando el resto como paso directo (identidad),
    para reducir el numero de compuertas AND-NOT-XOR activas por ronda."""
    out = [fila[:] for fila in state]
    for y in range(5):
        for b in bits_activos:
            for x in range(5):
                a_x = (state[x][y] >> b) & 1
                a_x1 = (state[(x + 1) % 5][y] >> b) & 1
                a_x2 = (state[(x + 2) % 5][y] >> b) & 1
                bit_salida = a_x ^ ((1 - a_x1) & a_x2)
                out[x][y] = (out[x][y] & ~(1 << b)) | (bit_salida << b)
    return out

print("\n[Punto 3] Intento 2 -- chi_parcial (cobertura parcial de bits, mitad del ancho):")
_bits_activos_64 = set(range(32))  # mitad activa (bits 0-31), mitad salteada (32-63)
_bit_salteado = 63

# Construimos, invirtiendo capa_lineal_rediseno, una diferencia con un unico bit
# activo exactamente en una posicion salteada por chi_parcial (bit 63), y
# confirmamos que la propagacion es DETERMINISTA (probabilidad 1, peso 0 en el
# sentido -log2(Pr)) sobre una muestra grande de pares aleatorios: al ser
# identidad en ese bit, la preservacion de la diferencia es un hecho algebraico
# (no depende del valor concreto), por eso una muestra grande basta para
# ilustrarlo sin necesidad de recorrer los 2^64 casos posibles.
import random
_rng = random.Random(99)
target = [[0] * 5 for _ in range(5)]
target[2][3] = 1 << _bit_salteado
_fallos = 0
for _ in range(20000):
    a = [[_rng.getrandbits(64) for _ in range(5)] for _ in range(5)]
    b = [[a[x][y] ^ target[x][y] for y in range(5)] for x in range(5)]
    out_a = chi_parcial(a, _bits_activos_64)
    out_b = chi_parcial(b, _bits_activos_64)
    diff = [[out_a[x][y] ^ out_b[x][y] for y in range(5)] for x in range(5)]
    if diff != target:
        _fallos += 1
print(f"  Diferencia en bit salteado (63) sobre 20000 pares aleatorios: "
      f"{20000 - _fallos}/20000 preservada exactamente (peso 0, Pr=1)")
assert _fallos == 0
print("  -> INVALIDO: introduce una caracteristica diferencial de 1 ronda con "
      "probabilidad 1 (peso 0), frente al minimo de peso 2 probado para chi estandar "
      "en MILP_AES_Keccak_modificado.ipynb. Se descarta.")

print(
    "\n[Punto 3] Conclusion: ambas simplificaciones de chi fallan (una rompe la "
    "biyectividad, la otra rompe la resistencia diferencial. Se mantiene chi "
    "ESTANDAR sin cambios; el analisis de costo de hardware, README.md, "
    "muestra que, implementado como una unica compuerta fusionada AND-NOT mas "
    "XOR, 2 compuertas/bit, profundidad 2, ya esta cerca del minimo posible para "
    "una funcion no lineal biyectiva de este tipo."
)


# ============================================================================
# Permutacion rediseñada completa: capa_lineal_rediseno + chi (estandar) + iota
# ============================================================================

def keccak_f1600_rediseno(state):
    """Variante estructural que responde a los puntos 2 y 3: reemplaza
    theta+rho+pi por capa_lineal_rediseno; mantiene chi estandar (unica opcion
    que preserva biyectividad y resistencia diferencial); conserva las rondas e
    iota dinamicas ya validadas (intentos_fallidos, dominio_aplicacion)."""
    rondas = min(24, 8 + intentos_fallidos)
    for round_idx in range(rondas):
        state = capa_lineal_rediseno(state)
        state = chi(state)
        state = iota(state, round_idx, intentos_fallidos, dominio_aplicacion)
    return state

def sha3_256_rediseno(message: bytes) -> bytes:
    """Variante de SHA3-256 usando keccak_f1600_rediseno (misma construccion
    esponja que sha3_256_variante)."""
    rate_bytes = 136
    state = [[0] * 5 for _ in range(5)]
    padded = bytearray(message)
    padded.append(0x06)
    while len(padded) % rate_bytes != 0:
        padded.append(0x00)
    padded[-1] |= 0x80
    for block_idx in range(0, len(padded), rate_bytes):
        block = padded[block_idx:block_idx + rate_bytes]
        for i in range(rate_bytes // 8):
            x = i % 5
            y = i // 5
            word = int.from_bytes(block[i*8:(i+1)*8], 'little')
            state[x][y] ^= word
        state = keccak_f1600_rediseno(state)
    output = bytearray()
    for i in range(4):
        x = i % 5
        y = i // 5
        output.extend(state[x][y].to_bytes(8, 'little'))
    return bytes(output)

hash_rediseno = sha3_256_rediseno(mensaje).hex()
print(f"\n[Estructural] Hash con keccak_f1600_rediseno (capa lineal nueva + chi estandar): {hash_rediseno}")
print(f"[Estructural] Hash original (sha3_256_fips):                                      {hash_original}")
print(f"[Estructural] Hash variante dinamica (sha3_256_variante):                         {hash_variante}")
assert hash_rediseno not in (hash_original, hash_variante), "el rediseno estructural debe producir un hash distinto"
print("Los tres hashes son distintos entre si -> el rediseno estructural es observable y no colisiona con las otras variantes.")


[Punto 2] capa_lineal_rediseno: rango GF(2) = 1600/1600 (INVERTIBLE, es una permutacion valida)

[Punto 2] Difusion a nivel de lana (fraccion de las 25 lanas tocadas), a 64 bits reales:


  rondas=1: REDISENO avg=100.0% min=100.0%  |  ESTANDAR (theta+rho+pi) avg=44.0% min=44.0%


  rondas=2: REDISENO avg=100.0% min=100.0%  |  ESTANDAR (theta+rho+pi) avg=100.0% min=100.0%


  rondas=3: REDISENO avg=100.0% min=100.0%  |  ESTANDAR (theta+rho+pi) avg=100.0% min=100.0%

[Punto 2] Resultado: capa_lineal_rediseno alcanza >=80% de difusion de lanas en UNA sola ronda (ver tabla), mientras que theta+rho+pi estandar necesita 2-3 rondas para llegar al mismo nivel, ver README.md para el analisis completo, incluyendo por que se descarto un primer intento mas simple (misma rotacion para toda la columna) que no escalaba con el ancho de palabra, y la comparacion de peso de trail diferencial en MILP_AES_Keccak_modificado.ipynb, que muestra una cota de resistencia igual en 1 ronda y notablemente MAYOR en 2-3 rondas frente al estandar.

[Punto 3] Intento 1 -- chi_sin_not (elimina el NOT, 1 compuerta menos por bit):
  chi estandar es biyectiva (fuerza bruta 32 casos): True
  chi_sin_not es biyectiva (fuerza bruta 32 casos):  False
  -> INVALIDO: al no ser biyectiva, keccak_f dejaria de ser una permutacion (colisiones internas de estado). Se descarta.

[Punto 3] Intento 2 -- 

  Diferencia en bit salteado (63) sobre 20000 pares aleatorios: 20000/20000 preservada exactamente (peso 0, Pr=1)
  -> INVALIDO: introduce una caracteristica diferencial de 1 ronda con probabilidad 1 (peso 0), frente al minimo de peso 2 probado para chi estandar en MILP_AES_Keccak_modificado.ipynb. Se descarta.

[Punto 3] Conclusion: ambas simplificaciones de chi fallan (una rompe la biyectividad, la otra rompe la resistencia diferencial. Se mantiene chi ESTANDAR sin cambios; el analisis de costo de hardware, README.md, muestra que, implementado como una unica compuerta fusionada AND-NOT mas XOR, 2 compuertas/bit, profundidad 2, ya esta cerca del minimo posible para una funcion no lineal biyectiva de este tipo.

[Estructural] Hash con keccak_f1600_rediseno (capa lineal nueva + chi estandar): 19a761f8107d2da3db543a3e25d5ec0cc53091bdcc852de14b878ff7e058c22a
[Estructural] Hash original (sha3_256_fips):                                      ab3fe4003f14e3ef573417f95e47d4985c482eadd139c08b37

In [5]:
# --- Objetivo 1: capacidad/ratio dependientes del dominio,
# y analisis de canal lateral ---
# Requiere haber ejecutado las celdas anteriores (PERFILES_DOMINIO,
# sha3_256_variante, keccak_f1600_variante, mensaje, hash_variante, etc.)

# ============================================================================
# Capacidad / ratio dependientes de dominio_aplicacion
# ============================================================================

print("[Punto 1] Perfiles (ratio, capacidad) por dominio_aplicacion:")
for dominio, perfil in PERFILES_DOMINIO.items():
    cota = cota_seguridad_generica_bits(perfil["capacidad_bits"])
    print(f"  dominio={dominio} ({perfil['nombre']}): rate={perfil['rate_bytes']*8} bits, "
          f"capacidad={perfil['capacidad_bits']} bits, cota de seguridad generica "
          f"= min(capacidad/2, 256) = {cota} bits"
          + (" *** POR DEBAJO de 256 bits ***" if cota < 256 else ""))

# Regresion: el dominio 0 (por defecto) NO debe cambiar el hash ya establecido
# (usa el mismo par rate=136/capacidad=512 que antes de este punto).
establecer_dominio_aplicacion(0)
hash_dominio0 = sha3_256_variante(mensaje).hex()
assert hash_dominio0 == HASH_BASELINE_SIN_DOMINIO, (
    "Regresion: el dominio 0 (rate=136, estandar) debe reproducir el hash baseline"
)
print(f"\n[Regresion] dominio_aplicacion=0 reproduce el hash baseline: {hash_dominio0}")

# Efecto observable del ratio/capacidad: mismo mensaje, mismo intentos_fallidos,
# 4 hashes distintos (uno por perfil rate/capacidad).
print("\n[Punto 1] Hash del mismo mensaje bajo los 4 perfiles de rate/capacidad:")
for dominio in PERFILES_DOMINIO:
    establecer_dominio_aplicacion(dominio)
    h = sha3_256_variante(mensaje).hex()
    print(f"  dominio={dominio}: {h}")
establecer_dominio_aplicacion(0)  # restaurar valor por defecto

print(
    "\n[Punto 1] Hallazgo critico: el dominio 1 (IoT/sensor) usa capacidad=448 bits, "
    "lo que da una cota de seguridad generica de 224 bits -- MENOR que los 256 bits "
    "nominales de la salida. Es una perdida de margen de seguridad CONOCIDA y "
    "cuantificada (no oculta): el perfil mas liviano (mayor rate, para hardware "
    "extremadamente limitado) paga ese costo en margen de seguridad generica, "
    "igual que SHA3-224 estandar. Los dominios 2 y 3 mantienen la cota completa "
    "de 256 bits (su capacidad ya excede 2*256)."
)


# ============================================================================
# Objetivo 1: analisis de canal lateral
# ============================================================================
# Dos vectores de canal lateral identificados en keccak_f1600_variante:
#  (a) el NUMERO DE RONDAS depende de intentos_fallidos -> tiempo de ejecucion
#      variable y correlacionado con esa variable.
#  (b) la RAMA if/else (orden clasico vs experimental) es una bifurcacion cuyo
#      camino de codigo depende de intentos_fallidos -> vector clasico de
#      canal lateral (tiempo/energia) en implementaciones no protegidas.
# Se demuestra (a) con una medicion real de tiempo de ejecucion.

import time

def _tiempo_promedio(intentos, repeticiones=200):
    global intentos_fallidos
    valor_previo = intentos_fallidos
    intentos_fallidos = intentos
    estado_inicial = [[0] * 5 for _ in range(5)]
    t0 = time.perf_counter()
    for _ in range(repeticiones):
        s = [fila[:] for fila in estado_inicial]
        keccak_f1600_variante(s)
    t1 = time.perf_counter()
    intentos_fallidos = valor_previo
    return (t1 - t0) / repeticiones

print("\n[Canal lateral] Tiempo promedio de keccak_f1600_variante segun intentos_fallidos:")
_tiempos = {}
for _intentos in (0, 2, 3, 5, 10, 16, 20):
    _t = _tiempo_promedio(_intentos)
    _tiempos[_intentos] = _t
    _rondas_esperadas = min(24, 8 + _intentos)
    print(f"  intentos_fallidos={_intentos:>2} (rondas={_rondas_esperadas:>2}): "
          f"{_t*1e6:.1f} microsegundos/llamada")

_creciente = all(_tiempos[a] <= _tiempos[b] * 1.15 for a, b in zip(list(_tiempos)[:-1], list(_tiempos)[1:]))
_correlacion_clara = _tiempos[20] > _tiempos[0] * 1.5
print(f"\n[Canal lateral] El tiempo crece de forma medible y aproximadamente monotona con "
      f"intentos_fallidos (t(20)/t(0) = {_tiempos[20]/_tiempos[0]:.2f}x): {_correlacion_clara}")
assert _correlacion_clara, "se esperaba una diferencia de tiempo claramente medible"

print(
    "\n[Canal lateral] CONCLUSION: si intentos_fallidos representa un contador de "
    "seguridad real (p. ej. intentos de login fallidos), esta correlacion tiempo<->"
    "intentos_fallidos es explotable por un atacante con acceso a medir tiempos de "
    "respuesta (ataque de temporizacion clasico, ej. Kocher 1996): permite inferir "
    "el valor del contador SIN acceso directo a el, y en particular detectar cuando "
    "el sistema esta operando con pocas rondas (intentos_fallidos bajo, la region ya "
    "identificada como de menor margen de seguridad en el informe. "
    "Esto agrava la debilidad ya conocida (rondas como estado mutable): no solo el "
    "margen de seguridad baja con intentos_fallidos bajo, sino que un atacante puede "
    "DETECTAR esa condicion por temporizacion sin necesitar acceso al contador. "
    "Un segundo vector (no medido aqui, pero estructuralmente presente): la rama "
    "if intentos_fallidos < 3 selecciona un camino de codigo distinto (con o sin "
    "sigma), lo que en hardware real (o en Python con JIT/cache) puede producir "
    "perfiles de tiempo y consumo energetico distinguibles para ese predicado "
    "especifico, independientemente del efecto agregado por numero de rondas. "
    "Esta implementacion NO aplica ninguna mitigacion de tiempo constante (por "
    "ejemplo, ejecutar siempre 24 rondas y descartar las de mas, o usar una "
    "seleccion de rama sin bifurcacion real) -- se documenta como limitacion "
    "conocida, no como algo resuelto."
)


[Punto 1] Perfiles (ratio, capacidad) por dominio_aplicacion:
  dominio=0 (general (por defecto)): rate=1088 bits, capacidad=512 bits, cota de seguridad generica = min(capacidad/2, 256) = 256 bits
  dominio=1 (IoT / sensor (recursos extremos)): rate=1152 bits, capacidad=448 bits, cota de seguridad generica = min(capacidad/2, 256) = 224 bits *** POR DEBAJO de 256 bits ***
  dominio=2 (tarjeta inteligente): rate=832 bits, capacidad=768 bits, cota de seguridad generica = min(capacidad/2, 256) = 256 bits
  dominio=3 (firmware embebido critico): rate=576 bits, capacidad=1024 bits, cota de seguridad generica = min(capacidad/2, 256) = 256 bits

[Regresion] dominio_aplicacion=0 reproduce el hash baseline: a5f8cd24080e6d56b242b7b9340d9c1af7274d41820e0a34b7cf906d9746cb33

[Punto 1] Hash del mismo mensaje bajo los 4 perfiles de rate/capacidad:
  dominio=0: a5f8cd24080e6d56b242b7b9340d9c1af7274d41820e0a34b7cf906d9746cb33
  dominio=1: 64a86d5cc3f491ac00b6a0084aee9ddcb7b75543b21d3811a217ff818c137828

  intentos_fallidos= 5 (rondas=13): 490.1 microsegundos/llamada
  intentos_fallidos=10 (rondas=18): 671.8 microsegundos/llamada


  intentos_fallidos=16 (rondas=24): 938.3 microsegundos/llamada
  intentos_fallidos=20 (rondas=24): 994.6 microsegundos/llamada

[Canal lateral] El tiempo crece de forma medible y aproximadamente monotona con intentos_fallidos (t(20)/t(0) = 4.89x): True

[Canal lateral] CONCLUSION: si intentos_fallidos representa un contador de seguridad real (p. ej. intentos de login fallidos), esta correlacion tiempo<->intentos_fallidos es explotable por un atacante con acceso a medir tiempos de respuesta (ataque de temporizacion clasico, ej. Kocher 1996): permite inferir el valor del contador SIN acceso directo a el, y en particular detectar cuando el sistema esta operando con pocas rondas (intentos_fallidos bajo, la region ya identificada como de menor margen de seguridad en el informe. Esto agrava la debilidad ya conocida (rondas como estado mutable): no solo el margen de seguridad baja con intentos_fallidos bajo, sino que un atacante puede DETECTAR esa condicion por temporizacion sin necesitar ac

In [6]:
# --- Viabilidad en hardware cuantificada: conteo real de compuertas y profundidad ---
# Requiere haber ejecutado las celdas anteriores (ROTATION_OFFSETS, ROTL64,
# theta, chi, capa_lineal_rediseno, etc.)
#
# Metodologia: se instrumenta cada operacion (XOR, AND, NOT) para que cuente
# explicitamente cuantas veces se ejecuta sobre palabras de 64 bits (cada
# ejecucion = 64 compuertas de 1 bit), en vez de solo describir el costo de
# palabra. NOT antes de un AND se cuenta como fusionado (compuerta AND-NOT, 1
# sola celda estandar de sintesis, consistente con lo ya explicado en README.md.

class _ContadorCompuertas:
    def __init__(self):
        self.xor = 0
        self.and_ = 0
        self.not_no_fusionado = 0
    def total(self):
        return self.xor + self.and_  # NOT fusionado no cuenta aparte

def _gxor(c, a, b, z=64):
    c.xor += z
    return a ^ b

def _gand(c, a, b, z=64):
    c.and_ += z
    return a & b

def _gnotfusionado(c, a, z=64):
    c.not_no_fusionado += z  # solo informativo: costo SI NO se fusionara con el AND
    return (~a) & 0xFFFFFFFFFFFFFFFF

def _theta_contado(state, c):
    state = [fila[:] for fila in state]
    C = [0] * 5
    for x in range(5):
        v = state[x][0]
        for y in range(1, 5):
            v = _gxor(c, v, state[x][y])
        C[x] = v
    D = [_gxor(c, C[(x - 1) % 5], ROTL64(C[(x + 1) % 5], 1)) for x in range(5)]
    for x in range(5):
        for y in range(5):
            state[x][y] = _gxor(c, state[x][y], D[x])
    return state

def _capa_lineal_rediseno_contada(state, c):
    state = [fila[:] for fila in state]
    C = [0] * 5
    for x in range(5):
        v = state[x][0]
        for y in range(1, 5):
            v = _gxor(c, v, state[x][y])
        C[x] = v
    D_base = [_gxor(c, C[(x - 1) % 5], ROTL64(C[(x + 1) % 5], 1)) for x in range(5)]
    for x in range(5):
        for y in range(5):
            state[x][y] = _gxor(c, state[x][y], ROTL64(D_base[x], ROTATION_OFFSETS[x][y]))
    E = [0] * 5
    for y in range(5):
        v = state[0][y]
        for x in range(1, 5):
            v = _gxor(c, v, state[x][y])
        E[y] = v
    F_base = [_gxor(c, E[(y - 1) % 5], ROTL64(E[(y + 1) % 5], 2)) for y in range(5)]
    for x in range(5):
        for y in range(5):
            state[x][y] = _gxor(c, state[x][y], ROTL64(F_base[y], ROTATION_OFFSETS[x][y]))
    return state

def _chi_contada(state, c):
    out = [[0] * 5 for _ in range(5)]
    for x in range(5):
        for y in range(5):
            notv = _gnotfusionado(c, state[(x + 1) % 5][y])
            andv = _gand(c, notv, state[(x + 2) % 5][y])
            out[x][y] = _gxor(c, state[x][y], andv)
    return out

_estado_prueba = [[(x * 5 + y + 1) * 0x1111111111 for y in range(5)] for x in range(5)]

c_theta = _ContadorCompuertas()
_theta_contado(_estado_prueba, c_theta)
c_rediseno = _ContadorCompuertas()
_capa_lineal_rediseno_contada(_estado_prueba, c_rediseno)
c_chi = _ContadorCompuertas()
_chi_contada(_estado_prueba, c_chi)

print("[Hardware] Compuertas por aplicacion (z=64, rho/pi no cuentan: rotacion/permutacion")
print("           por constante fija = solo cableado, 0 compuertas):")
print(f"  theta (estandar):        {c_theta.total():5d} compuertas  (theta+rho+pi = {c_theta.total():5d} total)")
print(f"  capa_lineal_rediseno:    {c_rediseno.total():5d} compuertas  "
      f"({c_rediseno.total()/c_theta.total():.2f}x el estandar)")
print(f"  chi (AND-NOT fusionado): {c_chi.total():5d} compuertas  "
      f"(NOT sin fusionar habria costado {c_chi.not_no_fusionado} compuertas extra)")

profundidad_theta = 5       # arbol XOR 5 entradas (3) + D (1) + aplicar (1)
profundidad_rediseno = 10   # dos pasadas secuenciales tipo theta (5+5)
profundidad_chi = 2         # AND-NOT fusionado (1) + XOR (1)

gates_ronda_estandar = c_theta.total() + c_chi.total()
gates_ronda_rediseno = c_rediseno.total() + c_chi.total()
profundidad_ronda_estandar = profundidad_theta + profundidad_chi
profundidad_ronda_rediseno = profundidad_rediseno + profundidad_chi

print(f"\n[Hardware] Costo por RONDA completa (capa lineal + chi; iota es 1 XOR de una")
print(f"           sola lana, despreciable):")
print(f"  ronda estandar (theta+rho+pi+chi):  {gates_ronda_estandar:5d} compuertas, profundidad {profundidad_ronda_estandar}")
print(f"  ronda rediseno (capa_lineal_rediseno+chi): {gates_ronda_rediseno:5d} compuertas, profundidad {profundidad_ronda_rediseno}")

print(
    "\n[Hardware] Comparacion de costo TOTAL para margen de seguridad equivalente, "
    "usando los pesos de trail de README.md 3.1/3.2 (z=8): rediseno a 2 rondas "
    "(peso 88) vs estandar a 3 rondas (peso 89) -- practicamente el mismo margen "
    "de resistencia diferencial:"
)
costo_rediseno_2r = 2 * gates_ronda_rediseno
costo_estandar_3r = 3 * gates_ronda_estandar
print(f"  rediseno x2 rondas:  {costo_rediseno_2r} compuertas-ronda")
print(f"  estandar x3 rondas:  {costo_estandar_3r} compuertas-ronda")
print(f"  razon: {costo_rediseno_2r/costo_estandar_3r:.2f}x "
      f"({'PRACTICAMENTE IGUAL' if 0.9 < costo_rediseno_2r/costo_estandar_3r < 1.1 else 'DISTINTO'})")

print(
    "\n[Hardware] Para z=4 la comparacion NO es tan favorable: el peso del rediseno a "
    "2 rondas (45) es MENOR que el del estandar a 3 rondas (62), por lo que a z=4 "
    "harian falta 3 rondas de rediseno (peso 103) para superar el margen del estandar "
    "a 3 rondas, a un costo de "
    f"{3*gates_ronda_rediseno} vs {costo_estandar_3r} compuertas-ronda "
    f"({(3*gates_ronda_rediseno)/costo_estandar_3r:.2f}x) -- una perdida neta de area/tiempo "
    "para ese ancho de palabra especifico."
)

print(
    "\n[Hardware] CONCLUSION HONESTA: capa_lineal_rediseno cuesta exactamente el doble "
    "de compuertas y el doble de profundidad que theta+rho+pi por ronda (verificado, no "
    "solo estimado). Su ventaja de difusion mas rapida (README 2.8) NO se traduce "
    "automaticamente en un ahorro neto de hardware: para z=8 el costo total termina "
    "practicamente empatado con el estandar a margen de seguridad equivalente, y para "
    "z=4 termina siendo PEOR. El beneficio real de capa_lineal_rediseno es de LATENCIA "
    "en regimenes de pocas rondas (alcanza buena difusion de lanas en 1 sola ronda, util "
    "si el numero de rondas esta limitado por diseno), no un ahorro de area/energia total "
    "garantizado. Esta implementacion tampoco reduce el numero de rondas "
    "(`rondas = min(24, 8+intentos_fallidos)` se mantiene igual para ambas variantes), "
    "por lo que el beneficio potencial de latencia NO se esta aprovechando todavia en el "
    "codigo actual -- queda como trabajo futuro ajustar la formula de rondas "
    "especificamente para capa_lineal_rediseno."
)


[Hardware] Compuertas por aplicacion (z=64, rho/pi no cuentan: rotacion/permutacion
           por constante fija = solo cableado, 0 compuertas):
  theta (estandar):         3200 compuertas  (theta+rho+pi =  3200 total)
  capa_lineal_rediseno:     6400 compuertas  (2.00x el estandar)
  chi (AND-NOT fusionado):  3200 compuertas  (NOT sin fusionar habria costado 1600 compuertas extra)

[Hardware] Costo por RONDA completa (capa lineal + chi; iota es 1 XOR de una
           sola lana, despreciable):
  ronda estandar (theta+rho+pi+chi):   6400 compuertas, profundidad 7
  ronda rediseno (capa_lineal_rediseno+chi):  9600 compuertas, profundidad 12

[Hardware] Comparacion de costo TOTAL para margen de seguridad equivalente, usando los pesos de trail de README.md 3.1/3.2 (z=8): rediseno a 2 rondas (peso 88) vs estandar a 3 rondas (peso 89) -- practicamente el mismo margen de resistencia diferencial:
  rediseno x2 rondas:  19200 compuertas-ronda
  estandar x3 rondas:  19200 compuertas-ronda
  ra

In [7]:
# --- Validacion con sintesis real de hardware (Yosys 0.33 + ABC) ---
# La celda anterior cuenta compuertas por OPERACION ejecutada en Python. Para
# validarlo con un metodo independiente se sintetizo cada componente como
# Verilog parametrizado y se sintetizo con Yosys+ABC (obtenido sin privilegios
# de administrador via `apt-get download` + `dpkg -x`, sin instalar), mapeando
# a compuertas AND/XOR/NAND/NOR/OR/AND-NOT y midiendo profundidad como el
# camino topologico mas largo real (comando `ltp`), no una estimacion de arbol
# balanceado. Los archivos .v y los logs de sintesis quedan en el scratchpad
# de la sesion; aqui se reportan los numeros obtenidos, ya que Yosys no es
# parte de este kernel de Python.

theta_gates_real, theta_depth_real = 3200, 6
chi_gates_real, chi_depth_real = 3200, 2
rediseno_gates_real, rediseno_depth_real = 6926, 12

ronda_estandar_gates_real = theta_gates_real + chi_gates_real
ronda_estandar_depth_real = theta_depth_real + chi_depth_real
ronda_rediseno_gates_real = rediseno_gates_real + chi_gates_real
ronda_rediseno_depth_real = rediseno_depth_real + chi_depth_real
razon_gates_real = ronda_rediseno_gates_real / ronda_estandar_gates_real

print("[Hardware, sintesis real] theta y chi coinciden EXACTAMENTE con el conteo por")
print("           operacion de la celda anterior (3200/3200 compuertas):")
print(f"  theta:                 {theta_gates_real:5d} compuertas, profundidad {theta_depth_real}")
print(f"  chi:                   {chi_gates_real:5d} compuertas, profundidad {chi_depth_real}")
print(f"  capa_lineal_rediseno:  {rediseno_gates_real:5d} compuertas, profundidad {rediseno_depth_real}  "
      f"(vs. {6400} compuertas / profundidad {10} del conteo por operacion: 8.2% mas cara y mas profunda,")
print("                         porque la sintesis expone que la etapa de filas depende")
print("                         secuencialmente de la salida completa de la etapa de columnas,")
print("                         no en paralelo como asumia la estimacion original de profundidad)")
print()
print(f"  ronda estandar (theta+rho+pi+chi):          {ronda_estandar_gates_real:5d} compuertas, profundidad {ronda_estandar_depth_real}")
print(f"  ronda rediseno (capa_lineal_rediseno+chi):  {ronda_rediseno_gates_real:5d} compuertas, profundidad {ronda_rediseno_depth_real}")
print(f"  razon de costo por ronda: {razon_gates_real:.3f}x (no es exactamente 1.5x ni 2.0x)")

# Tabla 4 del informe, ya corregida: la busqueda constructiva ampliada (probar
# los 31 patrones de fila no nulos como semilla, no solo los 5 de un bit)
# reduce la cota del estandar clasico porque "pi" cae entre la semilla y "chi"
# en ese orden y dispersa filas multi-bit en sub-filas mas baratas; no mejora
# ni a la variante dinamica ni al rediseno, que no tienen "pi" en esa posicion.
tabla4_corregida = {
    (4, 2): (8, 20, 45), (4, 3): (29, 70, 103), (4, 8): (315, 367, 393),
    (4, 17): (840, 901, 941), (4, 24): (1281, 1318, 1368),
    (8, 2): (8, 21, 88), (8, 3): (31, 93, 213), (8, 8): (578, 692, 810),
    (8, 17): (1655, 1787, 1890), (8, 24): (2521, 2624, 2743),
}

print()
print("[Hardware] Eficiencia peso-de-trail-por-compuerta, rediseno vs. estandar clasico corregido:")
print(f"{'z':>3} {'rondas':>6} {'peso estandar':>13} {'peso rediseno':>13} {'razon peso':>10} {'razon compuertas':>17} {'eficiente?':>10}")
for (z, r), (w_est, w_var, w_red) in sorted(tabla4_corregida.items()):
    razon_peso = w_red / w_est
    eficiente = "SI" if razon_peso > razon_gates_real else "no"
    print(f"{z:>3} {r:>6} {w_est:>13} {w_red:>13} {razon_peso:>10.3f} {razon_gates_real:>17.3f} {eficiente:>10}")

print()
print("[Hardware] CONCLUSION HONESTA, corregida: a 2 y 3 rondas capa_lineal_rediseno rinde")
print("           MAS margen de seguridad por compuerta que el estandar (razon de peso 3.6x a")
print("           11x, muy por encima de su costo extra de 1.582x). A partir de 8 rondas esa")
print("           ventaja se revierte (razon de peso 1.07x a 1.40x, ya no alcanza a justificar")
print("           el costo). El cruce ocurre entre la ronda 3 y la 8, para z=4 y z=8. La lectura")
print("           anterior de esta celda, basada en pesos de trail sin corregir y en el conteo")
print("           por operacion en vez de sintesis real, subestimaba la eficiencia real del")
print("           rediseno en regimenes de pocas rondas.")


[Hardware, sintesis real] theta y chi coinciden EXACTAMENTE con el conteo por
           operacion de la celda anterior (3200/3200 compuertas):
  theta:                  3200 compuertas, profundidad 6
  chi:                    3200 compuertas, profundidad 2
  capa_lineal_rediseno:   6926 compuertas, profundidad 12  (vs. 6400 compuertas / profundidad 10 del conteo por operacion: 8.2% mas cara y mas profunda,
                         porque la sintesis expone que la etapa de filas depende
                         secuencialmente de la salida completa de la etapa de columnas,
                         no en paralelo como asumia la estimacion original de profundidad)

  ronda estandar (theta+rho+pi+chi):           6400 compuertas, profundidad 8
  ronda rediseno (capa_lineal_rediseno+chi):  10126 compuertas, profundidad 14
  razon de costo por ronda: 1.582x (no es exactamente 1.5x ni 2.0x)

[Hardware] Eficiencia peso-de-trail-por-compuerta, rediseno vs. estandar clasico corregido:
  z rondas

In [7]:
# --- Experimentos adicionales, baratos y valiosos, sobre dominio_aplicacion ---
# Requiere haber ejecutado las celdas anteriores (PERFILES_DOMINIO,
# sha3_256_variante, establecer_dominio_aplicacion, etc.)

# ============================================================================
# Experimento 2: correccion del padding pad10*1 en los 4 perfiles de rate
# ============================================================================
# sha3_256_variante no puede compararse contra hashlib.sha3_256 (ese solo
# corre 24 rondas estandar; la variante siempre corre rondas=min(24,8+
# intentos_fallidos), aunque intentos_fallidos=0). Por eso esta prueba no
# compara hashes contra un oraculo externo: verifica una propiedad
# ESTRUCTURAL del padding (longitud multiplo del rate, y relleno minimo, sin
# desperdiciar un bloque entero de mas), replicando exactamente la misma
# logica de 4 lineas que usa sha3_256_variante, para los 4 valores reales de
# rate_bytes (no solo el estandar, rate=136, ya cubierto en el bloque 1).

def _padding_referencia(mensaje, rate_bytes):
    padded = bytearray(mensaje)
    padded.append(0x06)
    while len(padded) % rate_bytes != 0:
        padded.append(0x00)
    padded[-1] |= 0x80
    return bytes(padded)

print("[Experimento 2] Padding pad10*1 en fronteras de bloque, para los 4 perfiles de dominio_aplicacion:")
for dominio, perfil in PERFILES_DOMINIO.items():
    rate = perfil["rate_bytes"]
    for offset, etiqueta in ((-1, "rate-1"), (0, "rate"), (1, "rate+1")):
        largo = rate + offset
        if largo < 0:
            continue
        base = bytes(range(256))
        mensaje_prueba = (base * (largo // 256 + 1))[:largo]

        padded_ref = _padding_referencia(mensaje_prueba, rate)
        assert len(padded_ref) % rate == 0, (
            f"dominio={dominio} {etiqueta}: longitud rellenada no es multiplo de rate"
        )
        assert 0 < len(padded_ref) - len(mensaje_prueba) <= rate, (
            f"dominio={dominio} {etiqueta}: relleno desperdicia un bloque entero de mas"
        )

        establecer_dominio_aplicacion(dominio)
        hash1 = sha3_256_variante(mensaje_prueba)
        hash2 = sha3_256_variante(mensaje_prueba)
        assert len(hash1) == 32, f"dominio={dominio} {etiqueta}: salida no son 32 bytes"
        assert hash1 == hash2, f"dominio={dominio} {etiqueta}: sha3_256_variante no es determinista"

        print(f"  dominio={dominio} ({perfil['nombre']}), mensaje={largo}B ({etiqueta} de rate={rate}B): "
              f"padded={len(padded_ref)}B (multiplo de rate, relleno minimo), hash determinista de 32B OK")

establecer_dominio_aplicacion(0)
print("[Experimento 2] Padding correcto (multiplo de rate, relleno minimo, hash determinista) "
      "en los 3 casos de frontera para los 4 perfiles.")


# ============================================================================
# Experimento 3: canal lateral especifico de dominio_aplicacion (ratio/rate)
# ============================================================================
# A diferencia de intentos_fallidos (pensado como un contador de seguridad
# potencialmente sensible), dominio_aplicacion suele ser una eleccion de
# configuracion publica (que perfil de hardware/aplicacion se esta usando),
# no un secreto. Aun asi, se mide el canal lateral por completitud: para un
# MISMO mensaje e intentos_fallidos fijo, variar dominio_aplicacion cambia
# rate_bytes y por tanto el numero de bloques de esponja absorbidos.

mensaje_ch3 = bytes(500)  # mensaje de longitud fija para los 4 perfiles
intentos_fallidos = 0

print("\n[Experimento 3] Tiempo de sha3_256_variante segun dominio_aplicacion "
      "(mensaje fijo de 500 bytes, intentos_fallidos=0):")
_tiempos_dominio = {}
for dominio, perfil in PERFILES_DOMINIO.items():
    establecer_dominio_aplicacion(dominio)
    _bloques = -(-((len(mensaje_ch3) + 1)) // perfil["rate_bytes"])  # ceil, aproximado (sin contar el ultimo byte de padding)
    t0 = time.perf_counter()
    for _ in range(300):
        sha3_256_variante(mensaje_ch3)
    t1 = time.perf_counter()
    _t = (t1 - t0) / 300
    _tiempos_dominio[dominio] = _t
    print(f"  dominio={dominio} ({perfil['nombre']}), rate={perfil['rate_bytes']}B, "
          f"~{_bloques} bloques: {_t*1e6:.1f} microsegundos/llamada")

establecer_dominio_aplicacion(0)
_min_t = min(_tiempos_dominio.values())
_max_t = max(_tiempos_dominio.values())
print(f"\n[Experimento 3] Razon tiempo maximo/minimo entre perfiles: {_max_t/_min_t:.2f}x")
print(
    "[Experimento 3] CONCLUSION: dominio_aplicacion tambien deja huella medible en el tiempo "
    "de ejecucion (varia el numero de bloques de esponja para un mensaje de longitud fija), "
    "el mismo tipo de vector de canal lateral que intentos_fallidos. La diferencia practica es "
    "que dominio_aplicacion normalmente es una eleccion de configuracion PUBLICA (que perfil de "
    "hardware se usa), no un secreto que deba protegerse -- por eso este vector es de menor "
    "severidad que el de intentos_fallidos, pero se documenta por completitud en vez de asumir "
    "sin medir que no aplica."
)


[Experimento 2] Padding pad10*1 en fronteras de bloque, para los 4 perfiles de dominio_aplicacion:
  dominio=0 (general (por defecto)), mensaje=135B (rate-1 de rate=136B): padded=136B (multiplo de rate, relleno minimo), hash determinista de 32B OK
  dominio=0 (general (por defecto)), mensaje=136B (rate de rate=136B): padded=272B (multiplo de rate, relleno minimo), hash determinista de 32B OK
  dominio=0 (general (por defecto)), mensaje=137B (rate+1 de rate=136B): padded=272B (multiplo de rate, relleno minimo), hash determinista de 32B OK
  dominio=1 (IoT / sensor (recursos extremos)), mensaje=143B (rate-1 de rate=144B): padded=144B (multiplo de rate, relleno minimo), hash determinista de 32B OK
  dominio=1 (IoT / sensor (recursos extremos)), mensaje=144B (rate de rate=144B): padded=288B (multiplo de rate, relleno minimo), hash determinista de 32B OK
  dominio=1 (IoT / sensor (recursos extremos)), mensaje=145B (rate+1 de rate=144B): padded=288B (multiplo de rate, relleno minimo), hash d

  dominio=0 (general (por defecto)), rate=136B, ~4 bloques: 889.9 microsegundos/llamada


  dominio=1 (IoT / sensor (recursos extremos)), rate=144B, ~4 bloques: 1040.1 microsegundos/llamada


  dominio=2 (tarjeta inteligente), rate=104B, ~5 bloques: 1095.0 microsegundos/llamada


  dominio=3 (firmware embebido critico), rate=72B, ~7 bloques: 1492.7 microsegundos/llamada

[Experimento 3] Razon tiempo maximo/minimo entre perfiles: 1.68x
[Experimento 3] CONCLUSION: dominio_aplicacion tambien deja huella medible en el tiempo de ejecucion (varia el numero de bloques de esponja para un mensaje de longitud fija), el mismo tipo de vector de canal lateral que intentos_fallidos. La diferencia practica es que dominio_aplicacion normalmente es una eleccion de configuracion PUBLICA (que perfil de hardware se usa), no un secreto que deba protegerse -- por eso este vector es de menor severidad que el de intentos_fallidos, pero se documenta por completitud en vez de asumir sin medir que no aplica.


In [8]:
# --- Experimento de esfuerzo medio: combinacion nunca probada de
# capa_lineal_rediseno (puntos 2/3) con el comportamiento dinamico de sigma
# (punto 1), que hasta ahora eran caminos de codigo mutuamente excluyentes:
# keccak_f1600_variante usa theta/rho/pi estandar + intercambio chi/pi + sigma;
# keccak_f1600_rediseno usa capa_lineal_rediseno + chi SIN intercambio ni sigma.
# Aqui se construye keccak_f1600_rediseno_hibrido: capa_lineal_rediseno + chi,
# y ADEMAS sigma cuando intentos_fallidos >= 3 (no hay "pi" separado que
# intercambiar con chi en el rediseno, asi que el intercambio de orden no
# aplica; sigma si es una adicion independiente que puede sumarse).

def keccak_f1600_rediseno_hibrido(state):
    """capa_lineal_rediseno + chi (igual que keccak_f1600_rediseno), mas sigma
    cuando intentos_fallidos >= 3 (igual que el orden experimental de
    keccak_f1600_variante). Combinacion nunca probada hasta ahora."""
    rondas = min(24, 8 + intentos_fallidos)
    for round_idx in range(rondas):
        state = capa_lineal_rediseno(state)
        state = chi(state)
        if intentos_fallidos >= 3:
            state = sigma(state, intentos_fallidos)
        state = iota(state, round_idx, intentos_fallidos, dominio_aplicacion)
    return state

def sha3_256_rediseno_hibrido(message: bytes) -> bytes:
    """Variante de SHA3-256 usando keccak_f1600_rediseno_hibrido."""
    rate_bytes = PERFILES_DOMINIO[dominio_aplicacion]["rate_bytes"]
    state = [[0] * 5 for _ in range(5)]
    padded = bytearray(message)
    padded.append(0x06)
    while len(padded) % rate_bytes != 0:
        padded.append(0x00)
    padded[-1] |= 0x80
    for block_idx in range(0, len(padded), rate_bytes):
        block = padded[block_idx:block_idx + rate_bytes]
        for i in range(rate_bytes // 8):
            x = i % 5
            y = i // 5
            word = int.from_bytes(block[i*8:(i+1)*8], 'little')
            state[x][y] ^= word
        state = keccak_f1600_rediseno_hibrido(state)
    output = bytearray()
    for i in range(4):
        x = i % 5
        y = i // 5
        output.extend(state[x][y].to_bytes(8, 'little'))
    return bytes(output)

establecer_dominio_aplicacion(0)
intentos_fallidos = 5  # activa la rama con sigma tanto en variante como en el hibrido

hash_rediseno_sin_sigma = sha3_256_rediseno(mensaje).hex()
hash_hibrido = sha3_256_rediseno_hibrido(mensaje).hex()
hash_variante_con_sigma = sha3_256_variante(mensaje).hex()

print("[Experimento 5] Comparacion de las 3 variantes con intentos_fallidos=5 (rama con sigma activa):")
print(f"  rediseno SIN sigma:      {hash_rediseno_sin_sigma}")
print(f"  rediseno HIBRIDO (+sigma): {hash_hibrido}")
print(f"  variante (theta/rho/pi + sigma): {hash_variante_con_sigma}")

assert hash_hibrido != hash_rediseno_sin_sigma, "sigma deberia cambiar el resultado del rediseno"
assert hash_hibrido != hash_variante_con_sigma, "el hibrido no deberia coincidir con la variante estandar"
assert hash_hibrido == sha3_256_rediseno_hibrido(mensaje).hex(), "el hibrido debe ser determinista"

print(
    "\n[Experimento 5] Los 3 hashes son distintos entre si y el hibrido es determinista. "
    "sigma tiene un efecto real y observable tambien sobre capa_lineal_rediseno, "
    "consistente con lo ya visto para keccak_f1600_variante. La validacion de que esto no "
    "debilita la resistencia diferencial se hace en MILP_AES_Keccak_modificado.ipynb."
)


[Experimento 5] Comparacion de las 3 variantes con intentos_fallidos=5 (rama con sigma activa):
  rediseno SIN sigma:      19a761f8107d2da3db543a3e25d5ec0cc53091bdcc852de14b878ff7e058c22a
  rediseno HIBRIDO (+sigma): 07861fa6807da8c61d1a2301b810d8db6ea9de4a0d9557d03d0e8fbafa5c3d8e
  variante (theta/rho/pi + sigma): a5f8cd24080e6d56b242b7b9340d9c1af7274d41820e0a34b7cf906d9746cb33

[Experimento 5] Los 3 hashes son distintos entre si y el hibrido es determinista. sigma tiene un efecto real y observable tambien sobre capa_lineal_rediseno, consistente con lo ya visto para keccak_f1600_variante. La validacion de que esto no debilita la resistencia diferencial se hace en MILP_AES_Keccak_modificado.ipynb.
